In [ ]:
# Use the prepared FEVER diagnostic records and locally supplied model files.
from pathlib import Path
import json
import pandas as pd
from task_b.schema import parse_claim_evidence
from task_b.models import embedding_predictor, nli_predictor, model_fingerprint
from task_b.verification import timed_predictions, evaluate, fever_binary_proxy, save_run

PREPARED = Path("outputs/task_b/fever_baselines/prepared/records.jsonl")
EMBEDDING_DIR = Path("models/task_b/all-MiniLM-L6-v2")
NLI_DIR = Path("models/task_b/roberta-large-mnli")
OUTPUT = Path("outputs/task_b/embedding_nli")
assert PREPARED.is_file(), "Run task_b_fever_baselines.ipynb first"
assert EMBEDDING_DIR.is_dir() and NLI_DIR.is_dir(), "Place complete model files in the local directories"


In [ ]:
# Read the resolved claims without changing their gold labels or evidence.
rows = [json.loads(line) for line in PREPARED.read_text(encoding="utf-8").splitlines() if line.strip()]
examples = [parse_claim_evidence(row["claim"]) for row in rows]
gold = [row["gold_label"] for row in rows]
print("Resolved FEVER claims:", len(rows))


In [ ]:
# Load the embedding model locally and compare cosine similarities.
from sentence_transformers import SentenceTransformer
import sentence_transformers
embedding_model = SentenceTransformer(str(EMBEDDING_DIR.resolve()))
embedding_predictions, embedding_latency = timed_predictions(examples, embedding_predictor(embedding_model, threshold=0.5))
print("Embedding predictions complete")


In [ ]:
# Load local NLI weights; evidence is the premise and claim is the hypothesis.
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import transformers
import torch
nli_path = str(NLI_DIR.resolve())
tokenizer = AutoTokenizer.from_pretrained(nli_path, local_files_only=True)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_path, local_files_only=True)
nli_predictions, nli_latency = timed_predictions(examples, nli_predictor(tokenizer, nli_model, torch))
print("NLI predictions complete")


In [ ]:
# Save each method's config, raw predictions, and binary FEVER diagnostic separately.
results = {}
for name, predictions, latency, version, model_path in [("embedding", embedding_predictions, embedding_latency, sentence_transformers.__version__, EMBEDDING_DIR), ("nli", nli_predictions, nli_latency, transformers.__version__, NLI_DIR)]:
    binary = [fever_binary_proxy(label) for label in predictions]
    score = evaluate(gold, binary, ("SUPPORTED", "CONTRADICTED"), latency)
    score["raw_unsupported_count"] = predictions.count("UNSUPPORTED")
    score["evaluation_scope"] = "FEVER SUPPORTS/REFUTES gold-evidence diagnostic"
    results[name] = score
    save_run(OUTPUT / name, {"method": name, "prepared_records": str(PREPARED), "evaluation_source": rows[0]["source_file"], "model_path": str(model_path), "model_snapshot_sha256": model_fingerprint(model_path), "library_version": version, "model_config": "all-MiniLM-L6-v2 cosine threshold 0.5" if name == "embedding" else "roberta-large-mnli; project-specific sentence aggregation", "binary_proxy": "UNSUPPORTED -> CONTRADICTED for this diagnostic only"}, [{"claim_id": row["claim"]["claim_id"], "gold": row["gold_label"], "raw_prediction": prediction, "binary_prediction": projected, "latency_seconds": elapsed} for row, prediction, projected, elapsed in zip(rows, predictions, binary, latency)], score)


In [ ]:
# Compare concise metrics, per-class scores, and confusion matrices.
display(pd.DataFrame([{"method": name, "accuracy": score["accuracy"], "macro_f1": score["macro_f1"], "mean_latency_ms": score["latency_mean_ms"], "raw_unsupported": score["raw_unsupported_count"]} for name, score in results.items()]))
for name, score in results.items():
    print(name)
    display(pd.DataFrame(score["per_class"]).T)
    display(pd.DataFrame(score["confusion_matrix"], index=score["labels"], columns=score["labels"]))
